# Match Christian's neuron count on zplane02

**Goal:** reuse Christian's exact `data.bin` and `ops.npy` from `D:/jeff/cjennings/zplane02_tp00001-08440/` and re-run suite2p detection. If we can't reproduce his ROI count (1506 total / ~791 accepted) given identical inputs, the gap is traceable to a specific code-path or library-version difference rather than data preprocessing.

**Why bypass `lsp.pipeline`:** the LBM fork's `run_lsp.py:2334` overrides `ops['fs']` from TIFF metadata when `fs in (None, 10.0)`, which clobbers Christian's `fs=10` even when set explicitly. Bin size for detection is `round(tau*fs)` so this directly affects what cellpose sees. We call `suite2p.run_s2p.pipeline` directly to avoid that path.


In [ ]:
from pathlib import Path
import copy
import logging
import numpy as np
import torch

# Make suite2p / cellpose progress visible so the run is auditable.
logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s", force=True)

SRC = Path("D:/jeff/cjennings/zplane02_tp00001-08440")
OUT = Path("D:/jeff/cjennings/zplane02_replication")
OUT.mkdir(parents=True, exist_ok=True)

assert (SRC / "data.bin").exists(), f"missing data.bin at {SRC}"
assert (SRC / "ops.npy").exists(), f"missing ops.npy at {SRC}"
print(f"src: {SRC}")
print(f"out: {OUT}")

In [ ]:
# Load Christian's reference and print the targets we want to match.
ref_ops = np.load(SRC / "ops.npy", allow_pickle=True).item()
ref_stat = np.load(SRC / "stat.npy", allow_pickle=True)
ref_iscell = np.load(SRC / "iscell.npy")

print("=== Christian's reference ===")
print(f"  ROIs total   : {len(ref_stat)}")
print(f"  ROIs accepted: {int(ref_iscell[:, 0].sum())}")
print()
for k in ("fs", "tau", "diameter", "diameter_user", "anatomical_only",
          "threshold_scaling", "cellprob_threshold", "flow_threshold",
          "spatial_hp_cp", "spatial_scale", "high_pass", "max_overlap",
          "pretrained_model", "sparse_mode", "soma_crop", "smooth_sigma",
          "nbinned", "nframes", "Ly", "Lx"):
    v = ref_ops.get(k, "<missing>")
    if isinstance(v, np.ndarray):
        v = v.tolist() if v.size <= 4 else f"<arr {v.shape}>"
    print(f"  {k:22s} = {v}")

bin_size = int(max(1, ref_ops["nframes"] // ref_ops["nbinned"],
                   round(ref_ops["tau"] * ref_ops["fs"])))
print(f"\n  expected bin_size (round(tau*fs)) = {bin_size}")

In [ ]:
# Build a fresh ops dict pointing at the OUT dir but reading Christian's data.bin.
# We keep all his params (fs=10, diameter=6.08, threshold_scaling=1.0, ...) and
# clear only the detection outputs we want recomputed. Registration intermediates
# (meanImg, meanImgE, yrange, xrange, xoff, ...) are kept because do_registration=0
# means suite2p will read them from ops rather than recompute.
ops = copy.deepcopy(ref_ops)
ops["save_path"] = str(OUT)
ops["ops_path"] = str(OUT / "ops.npy")
ops["reg_file"] = str(SRC / "data.bin")           # ← reuse Christian's binary verbatim
ops["raw_file"] = str(SRC / "data_raw.bin")
ops["do_registration"] = 0
ops["roidetect"] = 1

for k in ("stat", "F", "Fneu", "spks", "iscell", "redcell",
         "detect_outputs", "nrois"):
    ops.pop(k, None)

# sanity: what's actually in ops['fs']
print(f"  ops['fs'] = {ops['fs']} (must stay 10.0 to match christian's bin_size)")

In [ ]:
# Translate fork ops -> upstream (db, settings) using the LBM fork's helper.
# The translator handles diameter list-form, baseline aliases, section nesting, etc.
# We then merge the derived settings on top of upstream's defaults to fill any keys
# the fork didn't propagate.
from suite2p.run_s2p import pipeline as upstream_pipeline
from suite2p import default_settings as us_default_settings
from suite2p.io.binary import BinaryFile
from lbm_suite2p_python.db_settings import ops_to_db_settings

_, derived = ops_to_db_settings(ops)
settings = us_default_settings()

def deep_merge(base, overlay):
    for k, v in overlay.items():
        if isinstance(v, dict) and isinstance(base.get(k), dict):
            deep_merge(base[k], v)
        else:
            base[k] = v
deep_merge(settings, derived)

print(f"settings['fs']        = {settings['fs']}")
print(f"settings['tau']       = {settings['tau']}")
print(f"settings['diameter']  = {settings['diameter']}")
print(f"detection algorithm   = {settings.get('detection', {}).get('algorithm')}")
cp = settings.get('detection', {}).get('cellpose_settings', {})
for k in ("flow_threshold", "cellprob_threshold", "highpass_spatial", "img", "cellpose_model"):
    print(f"cellpose.{k:18s} = {cp.get(k, '<missing>')}")

In [ ]:
# Save the prepped ops to OUT so suite2p has somewhere to write derived
# fields. We save BEFORE calling pipeline so a crash mid-run still leaves
# a recoverable ops on disk.
np.save(OUT / "ops.npy", ops, allow_pickle=True)
print(f"wrote {OUT / 'ops.npy'}")

# environment audit: anything that drifts here vs christian's run is a
# candidate for the result delta if counts don't match.
import suite2p
import cellpose
print()
print(f"suite2p file : {suite2p.__file__}")
print(f"cellpose file: {cellpose.__file__}")
print(f"torch        : {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"cuda device  : {torch.cuda.get_device_name(0)}")

In [ ]:
# Run upstream suite2p detection-only against christian's data.bin.
# This bypasses the LBM fork's fs-clobber entirely and uses the same code paths
# the fork would use after `_call_upstream_pipeline`.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Ly, Lx, n_frames = ops["Ly"], ops["Lx"], ops["nframes"]
print(f"opening: {ops['reg_file']}  Ly={Ly} Lx={Lx} n_frames={n_frames}")

with BinaryFile(Ly=Ly, Lx=Lx, filename=ops["reg_file"], n_frames=n_frames) as f_reg:
    (reg_outputs, detect_outputs, stat, F, Fneu, F_chan2, Fneu_chan2,
     spks, iscell, redcell, zcorr, plane_times) = upstream_pipeline(
        save_path=str(OUT),
        f_reg=f_reg,
        f_raw=None,
        f_reg_chan2=None,
        f_raw_chan2=None,
        run_registration=False,
        settings=settings,
        badframes=ops.get("badframes"),
        stat=None,
        device=device,
    )

In [ ]:
# Persist what we just produced (upstream pipeline already saved a few things
# to OUT, but be explicit about the canonical npy outputs).
np.save(OUT / "stat.npy", stat, allow_pickle=True)
np.save(OUT / "iscell.npy", iscell)
np.save(OUT / "F.npy", F)
np.save(OUT / "Fneu.npy", Fneu)
np.save(OUT / "spks.npy", spks)
if detect_outputs is not None:
    np.save(OUT / "detect_outputs.npy", detect_outputs, allow_pickle=True)
if reg_outputs is not None:
    np.save(OUT / "reg_outputs.npy", reg_outputs, allow_pickle=True)

# fold reg/detect outputs back into ops and re-save (matches the fork's contract)
if isinstance(reg_outputs, dict):
    ops.update(reg_outputs)
if isinstance(detect_outputs, dict):
    for k, v in detect_outputs.items():
        if k == "diameter" and v is not None:
            if isinstance(v, (list, tuple)) and len(v) >= 1:
                v = float(v[0]) if (len(v) < 2 or v[0] == v[1]) else (float(v[0]) + float(v[1])) / 2
        ops[k] = v
np.save(OUT / "ops.npy", ops, allow_pickle=True)
print(f"saved outputs to {OUT}")

In [ ]:
# Headline comparison.
n_ours = len(stat)
n_ours_accepted = int(iscell[:, 0].sum())
n_ref = len(ref_stat)
n_ref_accepted = int(ref_iscell[:, 0].sum())

print("=== detection comparison ===")
print(f"  total ROIs    : ours={n_ours:5d}   christian={n_ref:5d}   diff={n_ours - n_ref:+d}")
print(f"  accepted ROIs : ours={n_ours_accepted:5d}   christian={n_ref_accepted:5d}   diff={n_ours_accepted - n_ref_accepted:+d}")

# npix distribution comparison (very telling - tight median + small max means
# cellpose was splitting cells uniformly; sprawling distribution means raw masks)
ours_npix = np.array([s["npix"] for s in stat])
ref_npix = np.array([s["npix"] for s in ref_stat])
print()
print("=== npix distribution (all ROIs) ===")
for label, a in [("ours", ours_npix), ("christian", ref_npix)]:
    print(f"  {label:10s}: min={a.min():4d}  median={np.median(a):6.1f}  mean={a.mean():6.1f}  max={a.max():5d}")

In [ ]:
# Pixel-level comparison of the cellpose-input image. If max_proj matches
# but ROI count differs, the gap is in cellpose itself (model weights, GPU
# nondeterminism, version drift). If max_proj differs, the gap is in binning
# / high-pass / preprocessing.
if "max_proj" in ops and "max_proj" in ref_ops:
    mp_o = np.asarray(ops["max_proj"]).astype(np.float32)
    mp_r = np.asarray(ref_ops["max_proj"]).astype(np.float32)
    if mp_o.shape == mp_r.shape:
        diff = mp_o - mp_r
        corr = np.corrcoef(mp_o.ravel(), mp_r.ravel())[0, 1]
        print("=== max_proj (cellpose input for anatomical_only=4) ===")
        print(f"  ours      : mean={mp_o.mean():.2f}  std={mp_o.std():.2f}  min={mp_o.min():.2f}  max={mp_o.max():.2f}")
        print(f"  christian : mean={mp_r.mean():.2f}  std={mp_r.std():.2f}  min={mp_r.min():.2f}  max={mp_r.max():.2f}")
        print(f"  correlation: {corr:.6f}")
        print(f"  abs diff   : mean={np.abs(diff).mean():.2f}  max={np.abs(diff).max():.2f}")
    else:
        print(f"max_proj shape mismatch: {mp_o.shape} vs {mp_r.shape}")

In [ ]:
# Visual check: side-by-side max_proj.
import matplotlib.pyplot as plt

if "max_proj" in ops and "max_proj" in ref_ops:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    mp_o = np.asarray(ops["max_proj"])
    mp_r = np.asarray(ref_ops["max_proj"])
    vmax = max(mp_o.max(), mp_r.max())
    axes[0].imshow(mp_r, cmap="gray", vmax=vmax); axes[0].set_title(f"christian (n_rois={n_ref})")
    axes[1].imshow(mp_o, cmap="gray", vmax=vmax); axes[1].set_title(f"ours      (n_rois={n_ours})")
    axes[2].imshow(mp_o - mp_r, cmap="RdBu_r", vmin=-vmax/4, vmax=vmax/4); axes[2].set_title("diff (ours - christian)")
    for a in axes: a.axis("off")
    plt.tight_layout()
    plt.savefig(OUT / "max_proj_comparison.png", dpi=120)
    plt.show()
    print(f"saved {OUT / 'max_proj_comparison.png'}")